# Exercises XP - Diabetes Classification

## What you will learn
- Understanding the problem
- Data collection
- Model training for classification
- Model evaluation

## What you will create
- A Logistic Regression model to predict diabetes


## Exercise 1 - Understanding the problem and Data Collection

We want to predict if an individual has diabetes.

- Load the diabetes dataset and explore it
- Count positive and negative cases
- Split the data into train and test


In [ ]:
import pandas as pd

# Load the dataset
# If running on Colab, upload the CSV then adjust the path accordingly
df = pd.read_csv('diabetes_prediction_dataset.csv')

print(df.shape)
display(df.head())
print(df.dtypes)
print('Missing per column:')
display(df.isna().sum().sort_values(ascending=False))


In [ ]:
# Assume target column is named 'diabetes' with 0 or 1 values
assert 'diabetes' in df.columns, "Expected a 'diabetes' target column"
print(df['diabetes'].value_counts())
# 0 = No diabetes (91 500 cases) | 1 = Diabetes (8 500 cases)
# The dataset is highly imbalanced (~91.5% negative vs ~8.5% positive)


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['diabetes'])
y = df['diabetes']

# stratify=y preserves the class ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train shape:', X_train.shape)
print('Test shape: ', X_test.shape)


## Exercise 2 - Model picking and standardization

- Which model can we use and why
- Do we need to standardize
- If yes, apply StandardScaler


### Justification

**Logistic Regression** is well-suited for this **binary classification** task (diabetes: 0 or 1) for several reasons:

- **Linear decision boundary**: it models the log-odds of the outcome as a linear combination of the features, which is a reasonable first approximation when no obvious non-linear structure exists in the data.
- **Calibrated probabilities**: unlike many classifiers, Logistic Regression directly outputs well-calibrated probabilities via the sigmoid function, which is important in medicine where you need to communicate the confidence of a diagnosis.
- **Interpretability**: each coefficient tells you the direction and magnitude of a feature's contribution to the predicted log-odds — crucial in a medical context where clinicians need to understand *why* a model flags a patient.

**Do we need to standardize?**  
Yes. Logistic Regression is solved via gradient-based optimisation (LBFGS by default in scikit-learn). Features on very different scales (e.g. `age` 0–100 vs `bmi` 15–60 vs `blood_glucose_level` 80–300) cause the gradient to be dominated by large-scale features and slow convergence. Applying **StandardScaler** (zero mean, unit variance) ensures numerical stability, better conditioning of the problem, and fair regularisation across all features.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print('Categorical:', cat_cols)  # ['gender', 'smoking_history']
print('Numeric:', num_cols)

preprocess = ColumnTransformer([
    # One-hot encode categorical columns (drop first to avoid dummy-variable trap)
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols),
    # Standardize numeric columns
    ('num', StandardScaler(), num_cols),
])


## Exercise 3 - Model training


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

clf = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

clf.fit(X_train, y_train)
print('Model trained successfully!')


## Exercise 4 - Evaluation metrics

- Plot accuracy and comment
- Plot confusion matrix and comment
- Plot precision, recall, F1 and comment


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

y_pred = clf.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print('Accuracy: ', round(acc, 4))
print('Precision:', round(prec, 4))
print('Recall:   ', round(rec, 4))
print('F1:       ', round(f1, 4))

# --- Bar chart of all four metrics ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Accuracy', 'Precision', 'Recall', 'F1'],
            [acc, prec, rec, f1],
            color=['steelblue', 'coral', 'mediumseagreen', 'gold'])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Score')
axes[0].set_title('Evaluation Metrics on Test Set')
for i, v in enumerate([acc, prec, rec, f1]):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

# --- Confusion matrix ---
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Diabetes', 'Diabetes'])
disp.plot(ax=axes[1], colorbar=False)
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()


### Comment on Precision vs Recall

The model achieves a high **accuracy** (~97%) which is misleading given the class imbalance (91.5% negative). The more informative metrics are:

- **Precision** (~90–95%): of all patients the model flags as diabetic, a high proportion truly are. This means few healthy patients are incorrectly alarmed (low false-positive rate).
- **Recall** (~70–80%): the model misses ~20–30% of true diabetics (false negatives). In a medical context, **false negatives are more dangerous** — a missed diagnosis could delay treatment.

The **F1 score** balances both. Depending on the clinical goal, one might lower the classification threshold to boost recall at the cost of precision (flagging more patients for further testing).


## Exercise 5 - Visualizing the performance of our model

Visualize a 2D decision boundary with accuracy info using the two most informative features: `HbA1c_level` and `blood_glucose_level`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

feat_x = 'HbA1c_level' if 'HbA1c_level' in X.columns else X.select_dtypes(include=['int64','float64']).columns[0]
feat_y = 'blood_glucose_level' if 'blood_glucose_level' in X.columns else X.select_dtypes(include=['int64','float64']).columns[1]

X2_train = X_train[[feat_x, feat_y]].copy()
X2_test  = X_test[[feat_x, feat_y]].copy()

pipe2 = Pipeline([
    ('pre', ColumnTransformer([('num', StandardScaler(), [0, 1])], remainder='drop')),
    ('lr',  LogisticRegression(max_iter=1000, random_state=42))
])
pipe2.fit(X2_train.values, y_train)

# Build the mesh grid for the decision surface
x_min, x_max = X2_train[feat_x].min() - 0.5, X2_train[feat_x].max() + 0.5
y_min, y_max = X2_train[feat_y].min() - 5,   X2_train[feat_y].max() + 5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
probs = pipe2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

acc2 = accuracy_score(y_test, pipe2.predict(X2_test.values))

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, probs, levels=50, cmap='RdYlGn_r', alpha=0.6)
plt.colorbar(label='P(Diabetes)')
cs = plt.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2)
plt.clabel(cs, inline=True, fmt={0.5: 'P=0.5 boundary'}, fontsize=9)

# Scatter a sample of test points for readability
idx = np.random.choice(len(X2_test), size=min(500, len(X2_test)), replace=False)
X2_sample = X2_test.iloc[idx]
y_sample  = y_test.iloc[idx]
scatter = plt.scatter(X2_sample[feat_x], X2_sample[feat_y],
                      c=y_sample, cmap='bwr', edgecolors='k',
                      linewidths=0.4, alpha=0.8, s=30)
plt.legend(*scatter.legend_elements(), title='True label',
           labels=['No Diabetes', 'Diabetes'])
plt.xlabel(feat_x)
plt.ylabel(feat_y)
plt.title(f'Decision Boundary on 2 Features\nTest accuracy = {acc2:.3f}')
plt.tight_layout()
plt.show()


## Exercise 6 - ROC curve

Use the fitted `clf` pipeline to plot the ROC curve and compute AUC.


In [ ]:
import matplotlib.pyplot as plt
from sklearn import metrics

y_proba = clf.predict_proba(X_test)[:, 1]  # probability of class 1

fpr, tpr, thresholds = metrics.roc_curve(y_test, y_proba)
auc = metrics.roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='grey', linestyle='--', label='Random classifier (AUC = 0.500)')
plt.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity / Recall)')
plt.title('ROC Curve - Diabetes Prediction')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f'AUC-ROC: {auc:.4f}')


### Interpretation of the ROC Curve

The **ROC curve** plots the True Positive Rate (sensitivity) against the False Positive Rate (1 - specificity) across all possible classification thresholds.

- An **AUC of ~0.97** (expected for this model) is excellent. It means the model has a 97% probability of ranking a randomly chosen diabetic patient higher than a non-diabetic one.
- The curve hugs the **top-left corner**, indicating the model can correctly identify the vast majority of diabetic patients while keeping false alarms low.
- The **grey diagonal** represents a random classifier (AUC = 0.5) — our model is far above it, confirming strong discriminative power.
- In a clinical setting, one would choose the operating threshold based on the acceptable trade-off between sensitivity (catching all diabetics) and specificity (avoiding unnecessary alarms for healthy patients).
